# 2. Treinamento e avaliação
Divide os dados estratificadamente em treino (60%), validação (20%) e teste (20%). Compara modelos usando CV somente no treino, escolhe threshold F2 na validação e avalia o modelo final uma única vez no teste.


In [1]:
import json
import joblib
import pandas as pd
from sklearn.metrics import f1_score, fbeta_score, precision_score, recall_score
from breast_cancer.config import (
    BEST_MODEL_DIR, CANDIDATE_MODEL_DIR, DATA_PATH, FIGURE_DIR,
    OOF_REPORT_DIR, REPORT_DIR, THRESHOLD_BETA, VALIDATION_REPORT_DIR,
)
from breast_cancer.data import load_data
from breast_cancer.preprocessing import prepare_data
from breast_cancer.models import make_random_forest, make_xgboost
from breast_cancer.validation import (
    cross_validated_probabilities, cross_validation_scores,
    select_threshold, threshold_analysis,
)
from breast_cancer.evaluation import evaluate_model
from breast_cancer.prediction import load_model_artifact, predict_proba
from breast_cancer.visualization import (
    save_confusion_matrix, save_pr_curve, save_roc_curve,
)

(X_train, X_validation, X_test,
 y_train, y_validation, y_test) = prepare_data(load_data(DATA_PATH))
print(f'Treino: {X_train.shape}; validação: {X_validation.shape}; teste: {X_test.shape}')


Treino: (341, 30); validação: (114, 30); teste: (114, 30)


In [2]:
xgboost = make_xgboost()
xgb_cv = cross_validation_scores(xgboost, X_train, y_train, cv=3, scoring='accuracy')
print('XGBoost | CV accuracy (3 folds, treino):', xgb_cv)


XGBoost | CV accuracy (3 folds, treino): [0.95614035 0.94736842 0.94690265]


In [3]:
forest = make_random_forest()
forest_cv = cross_validation_scores(forest, X_train, y_train, cv=3, scoring='accuracy')
print('Random Forest | CV accuracy (3 folds, treino):', forest_cv)


Random Forest | CV accuracy (3 folds, treino): [0.95614035 0.94736842 0.92035398]


In [4]:
candidate_cv = {'xgboost': xgb_cv, 'random_forest': forest_cv}
best_model_name = max(candidate_cv, key=lambda name: candidate_cv[name].mean())
candidate_models = {'xgboost': xgboost, 'random_forest': forest}
best_model = candidate_models[best_model_name]

# Ajusta os candidatos apenas no treino; a validação escolhe threshold F2.
for candidate in candidate_models.values():
    candidate.fit(X_train, y_train)

validation_probabilities = predict_proba(best_model, X_validation)
threshold = select_threshold(y_validation, validation_probabilities, beta=THRESHOLD_BETA)
validation_metrics = evaluate_model(y_validation, validation_probabilities, threshold)

# OOF estima o desempenho no treino sem usar a linha durante aquele ajuste.
oof_probabilities = cross_validated_probabilities(
    make_xgboost() if best_model_name == 'xgboost' else make_random_forest(),
    X_train, y_train, cv=5,
)
oof_metrics = evaluate_model(y_train, oof_probabilities, threshold)

# O teste só é consultado depois de modelo e threshold estarem definidos.
test_probabilities = predict_proba(best_model, X_test)
test_metrics = evaluate_model(y_test, test_probabilities, threshold)

candidate_cv_means = {name: float(scores.mean()) for name, scores in candidate_cv.items()}
best_model_path = BEST_MODEL_DIR / f'{best_model_name}_best.joblib'
for directory in (CANDIDATE_MODEL_DIR, BEST_MODEL_DIR, REPORT_DIR,
                  OOF_REPORT_DIR, VALIDATION_REPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

joblib.dump(candidate_models['xgboost'], CANDIDATE_MODEL_DIR / 'xgboost.joblib')
joblib.dump(candidate_models['random_forest'], CANDIDATE_MODEL_DIR / 'random_forest.joblib')
joblib.dump({
    'model': best_model,
    'threshold': threshold,
    'feature_names': list(X_train.columns),
    'model_name': best_model_name,
    'threshold_metric': 'f_beta',
    'threshold_beta': THRESHOLD_BETA,
}, best_model_path)

oof_predictions = pd.DataFrame({
    'row_index': X_train.index,
    'y_true': y_train.to_numpy(),
    'probability_malignant': oof_probabilities,
    'predicted_label': (oof_probabilities >= threshold).astype(int),
})
oof_predictions.to_csv(OOF_REPORT_DIR / 'oof_predictions.csv', index=False)
threshold_table = threshold_analysis(
    y_validation, validation_probabilities, beta=THRESHOLD_BETA,
)
threshold_table.to_csv(VALIDATION_REPORT_DIR / 'threshold_analysis.csv', index=False)

save_roc_curve(
    test_metrics['fpr'], test_metrics['tpr'], test_metrics['auc_roc'],
    FIGURE_DIR / 'curva_roc.png',
)
save_pr_curve(
    test_metrics['precision_curve'], test_metrics['recall_curve'], test_metrics['auprc'],
    FIGURE_DIR / 'curva_pr.png',
)
save_confusion_matrix(y_test, test_metrics['predicted_labels'], FIGURE_DIR / 'matriz_confusao.png')

def malignant_metrics(y_true, metrics):
    predictions = metrics['predicted_labels']
    return {
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'f_beta': float(fbeta_score(y_true, predictions, beta=THRESHOLD_BETA, zero_division=0)),
    }

summary = {
    'split': {'train': len(y_train), 'validation': len(y_validation), 'test': len(y_test)},
    'best_model': best_model_name,
    'selection_metric': 'mean_cv_accuracy_on_train',
    'threshold': float(threshold),
    'threshold_selection_metric': 'validation_f_beta',
    'threshold_beta': THRESHOLD_BETA,
    'cv_accuracy': {name: scores.tolist() for name, scores in candidate_cv.items()},
    'oof': {
        'folds': 5,
        'auc_roc': oof_metrics['auc_roc'],
        'auprc': oof_metrics['auprc'],
        'at_validation_threshold': malignant_metrics(y_train, oof_metrics),
        'accuracy': oof_metrics['accuracy'],
        'confusion_matrix': oof_metrics['confusion_matrix'].tolist(),
    },
    'validation': {
        'auc_roc': validation_metrics['auc_roc'],
        'auprc': validation_metrics['auprc'],
        'at_selected_threshold': malignant_metrics(y_validation, validation_metrics),
        'accuracy': validation_metrics['accuracy'],
        'confusion_matrix': validation_metrics['confusion_matrix'].tolist(),
        'classification_report': validation_metrics['classification_report'],
    },
    'test': {
        'auc_roc': test_metrics['auc_roc'],
        'auprc': test_metrics['auprc'],
        'at_selected_threshold': malignant_metrics(y_test, test_metrics),
        'accuracy': test_metrics['accuracy'],
        'confusion_matrix': test_metrics['confusion_matrix'].tolist(),
        'classification_report': test_metrics['classification_report'],
    },
}
(REPORT_DIR / 'metrics.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
print(f'Melhor modelo por CV no treino: {best_model_name} ({candidate_cv_means[best_model_name]:.3f})')
print(f'Threshold F2 escolhido na validação: {threshold:.4f}')
print(f"Validação | recall: {validation_metrics['classification_report'].splitlines()[3].split()[2]}; F1: {validation_metrics['classification_report'].splitlines()[3].split()[3]}")
print(f"Teste intocado | AUC-ROC: {test_metrics['auc_roc']:.4f}; AUPRC: {test_metrics['auprc']:.4f}; accuracy: {test_metrics['accuracy']:.4f}")
print(test_metrics['classification_report'])
print(f'Resultados salvos em {REPORT_DIR}; modelo final: {best_model_path}')


Melhor modelo por CV no treino: xgboost (0.950)
Threshold F2 escolhido na validação: 0.2247
Validação | recall: 1.000; F1: 0.989
Teste intocado | AUC-ROC: 0.9947; AUPRC: 0.9934; accuracy: 0.9737
              precision    recall  f1-score   support

     Benigno      0.986     0.972     0.979        72
     Maligno      0.953     0.976     0.965        42

    accuracy                          0.974       114
   macro avg      0.970     0.974     0.972       114
weighted avg      0.974     0.974     0.974       114

Resultados salvos em C:\Users\mateu_g1p9kd2\OneDrive\Documentos\GitHub\Predicao-cancer-de-mama-vs2\reports; modelo final: C:\Users\mateu_g1p9kd2\OneDrive\Documentos\GitHub\Predicao-cancer-de-mama-vs2\models\best_model\xgboost_best.joblib


## Inferência com o modelo selecionado
O artefato contém o modelo ajustado no treino, o threshold selecionado na validação e a ordem das features esperadas.


In [5]:
artifact = load_model_artifact(best_model_path)
predict_proba(
    artifact['model'], X_test, feature_names=artifact['feature_names']
)[:3]


array([0.01506511, 0.0614743 , 0.02449833], dtype=float32)